# Install Libraries

In [1]:
pip install google-generativeai

  Obtaining dependency information for google-generativeai from https://files.pythonhosted.org/packages/b5/7f/35f89209487f8473edc9d2cecef894a54680cf666e32893a767d12a8dba9/google_generativeai-0.3.2-py3-none-any.whl.metadata
  Obtaining dependency information for google-ai-generativelanguage==0.4.0 from https://files.pythonhosted.org/packages/40/c2/d28988d3cba74e712f47a498e2b3e3b58ac215106019bf5d8c20f8ab9822/google_ai_generativelanguage-0.4.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.9/146.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 598.7/598.7 kB 8.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# Load Model

In [2]:
import time

In [3]:
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")
genai.configure(api_key = api_key)

# Generate content
model = genai.GenerativeModel(model_name='gemini-pro')
#response = model.generate_content('Tell me a story about a magic backpack.')
#print(response.text)

In [4]:
def generate_entities (prompt):
    response = model.generate_content(prompt)
    return response.text

In [5]:
def generate_prompt (text,info):
    if info == "CV":
        prompt = '''
        You are a entity extraction expert, you can identify and extract dfferent types of entities from a text. Here is some information from a CV. Your task is to find and enlist all the information entities like education (degree, grade, school name) skills (which skills he has), qualifications (skills), experience (action verb and nouns) and any other helpful token that is important for job, and share them in a list where entities are seperated by comma.  do not write anything else.  Don't share details. just the small entities seperated by comma in a dictonary (json). Each entity can have only 1-2 words..>>'''
        prompt = prompt+"```"+text+"```"
    else:
        prompt = '''
        You are a entity extraction expert, you can identify and extract dfferent types of entities from a text. Here is some information from a job description. Your task is to find and enlist all the information entities like education (degree requirement) skills (which skills the job needs), qualifications (skills), experience (action verb and nouns) and any other helpful token that is important for job, and share them in a list where entities are seperated by comma.  do not write anything else. Don't share details. just the small entities seperated by comma  in a dictonary (json). Each entity can have only 1-2 words.   Don't share details. >>'''
        prompt = prompt+"```"+text+"```"
    prompt = prompt + '''
    =================
    Example:
    {
        'Education': ['ABC University', 'CGPA 3.00', 'Computer Science and Engineering', 'BSc'],
        'Skills' : ['C','Python','R','Machine Learning', 'Communication', 'Team Work']
        'Experience' : {'ABX InfoTech': ['Team Management', 'Assistant Manager'] , 'STech':['Manager','Senior Enginner','AWS']}
    }
    '''
    return prompt

In [6]:
import pandas as pd

In [7]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/nlp4hr/Selected JDs.csv
/kaggle/input/nlp4hr/Selected CVs.csv
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00002-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer_config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model.bin.index.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00001-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/special_tokens_map.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/.gitattributes
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.model
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/generation_config.json


In [8]:
JD = pd.read_csv("/kaggle/input/nlp4hr/Selected JDs.csv")
CV = pd.read_csv("/kaggle/input/nlp4hr/Selected CVs.csv")

In [9]:
outputs_JD = []
outputs_CV = []
for i in range (200):
    JD_i = JD['jobpost'][i]
    CV_i = CV['Resume_str'][i]
    
    print ("Tring: ",i+1)
    
    try:
        JD_t = generate_entities(generate_prompt (JD_i,"JD"))
        print("JD :",JD_t)
    except:
        print("JD Failed")
        
    try:
        CV_t = generate_entities(generate_prompt (CV_i,"CV"))
        print("CV :",CV_t)
    except:
        print("CV Failed")
    
    outputs_JD.append(JD_t)
    outputs_CV.append(CV_t)  
    
    time.sleep(10)

Tring:  1
JD : ```json
{
  'Education': ['Higher Technical (Engineering)'],
  'Experience': ['start-up', 'commissioning', 'installation', 'service', 'management'],
  'Skills': ['Electronics', 'Electro-mechanics', 'Thermodynamics', 'Communication', 'Sales', 'Responsibility', 'Computer literacy', 'Armenian', 'Russian', 'English']
}
```
CV : ```json
{
  'Skills': [
    'Enterprise platforms',
    'Product Lifecycle Management (PLM)',
    'Project tracking',
    'Hardware and software upgrade planning',
    'Product requirements documentation',
    'Self-directed',
    'MS Visio',
    'Decisive',
    'Collaborative',
    'Domain Active Directory Layout',
    'Data storage engineering',
    'Information Assurance',
    'Risk Management Framework (RMF)',
    'Active Directory design and deployment',
    'Workstation build and deployment',
    'Systems Accreditation Packages',
    'Red Hat Enterprise Linux installation and hardening',
    'Network Design & Troubleshooting',
    'High Performa

In [10]:
dfx = pd.DataFrame({
    'Outputs_JD': outputs_JD,
    'Outputs_CV': outputs_CV
})

In [11]:
dfx.to_csv("outputs.csv")